<a href="https://colab.research.google.com/github/its-rohit-yadav/HyperparameterTunning-/blob/main/Hyperparameter%20Tunning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense, Dropout
from keras.callbacks import EarlyStopping

In [2]:
!pip install -q keras-tuner
import keras_tuner as kt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 5.7 MB/s eta 0:00:00


In [3]:
df = pd.read_csv('diabetes.csv')

print(f"Shape: {df.shape}")
print(f"\nClass distribution:\n{df['Outcome'].value_counts()}")
print(f"\nClass balance: {df['Outcome'].mean():.2%} positive (diabetic)")
df.head()

Shape: (768, 9)

Class distribution:
Outcome
0    500
1    268
Name: count, dtype: int64

Class balance: 34.90% positive (diabetic)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [5]:
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in zero_cols:
    n_zeros = (df[col] == 0).sum()
    print(f"{col}: {n_zeros} zero values ({n_zeros/len(df):.1%})")

Glucose: 5 zero values (0.7%)
BloodPressure: 35 zero values (4.6%)
SkinThickness: 227 zero values (29.6%)
Insulin: 374 zero values (48.7%)
BMI: 11 zero values (1.4%)


In [6]:
for col in zero_cols:
    median_val = df[col][df[col] != 0].median()
    df[col] = df[col].replace(0, median_val)

print("Zero values replaced with column medians.")

Zero values replaced with column medians.


In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = df.drop('Outcome', axis=1)
y = df['Outcome']
sc = StandardScaler()
X_scaled = sc.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")
print(f"Features: {X_train.shape[1]}")

Train size: 614 | Test size: 154
Features: 8


In [8]:
def build_baseline():
    model = Sequential([
        Dense(32, activation='relu', input_dim=8),
        Dropout(0.3),
        Dense(16, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

baseline_model = build_baseline()
baseline_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 833 (3.25 KB)

 Trainable params: 833 (3.25 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,        # stop if val_loss doesn't improve for 10 consecutive epochs
    restore_best_weights=True
)

baseline_history = baseline_model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 141ms/step - accuracy: 0.6710 - loss: 0.6266 - val_accuracy: 0.7338 - val_loss: 0.5808
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7362 - loss: 0.5858 - val_accuracy: 0.7403 - val_loss: 0.5444
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7280 - loss: 0.5683 - val_accuracy: 0.7403 - val_loss: 0.5199
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7329 - loss: 0.5290 - val_accuracy: 0.7468 - val_loss: 0.5054
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7508 - loss: 0.5050 - val_accuracy: 0.7468 - val_loss: 0.4972
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7541 - loss: 0.5083 - val_accuracy: 0.7403 - val_loss: 0.4958
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7443 - loss: 0.4931 - val_accuracy: 0.7338 - val_loss: 0.4973
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7834 - loss: 0.4756 - val_accuracy: 0.7273 -

In [11]:
baseline_loss, baseline_acc = baseline_model.evaluate(X_test, y_test, verbose=0)
print(f"Baseline Test Accuracy: {baseline_acc:.4f}")

Baseline Test Accuracy: 0.7403


In [13]:
#Tunning the optimizer
def build_model_optimizer(hp):
    model = Sequential()
    model.add(Dense(32, activation='relu', input_dim=8))
    model.add(Dropout(0.3))
    model.add(Dense(16, activation='relu'))
    model.add(Dropout(0.2))
    model.add(Dense(1, activation='sigmoid'))

    optimizer = hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop'])

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model



In [14]:
tuner_optimizer = kt.RandomSearch(
    build_model_optimizer,
    objective='val_accuracy',
    max_trials=5,
    seed=42,
    directory='tuner_optimizer',
    project_name='exp1'
)

In [15]:
tuner_optimizer.search(
    X_train, y_train,
    epochs=20,
    validation_data=(X_test, y_test),
    callbacks=[EarlyStopping(monitor='val_loss', patience=5)],
    verbose=0
)



In [16]:
best_optimizer_hp = tuner_optimizer.get_best_hyperparameters()[0]
print(f"Best optimizer: {best_optimizer_hp.values}")

Best optimizer: {'optimizer': 'rmsprop'}


In [17]:
model_exp1 = tuner_optimizer.get_best_models(num_models=1)[0]

history_exp1 = model_exp1.fit(
    X_train, y_train,
    epochs=100,
    validation_data=(X_test, y_test),
    callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
    initial_epoch=20,  # continue from where the tuner left off
    verbose=1
)

loss1, acc1 = model_exp1.evaluate(X_test, y_test, verbose=0)
print(f"Exp 1 (Optimizer Tuning) Test Accuracy: {acc1:.4f}")

Epoch 21/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 97ms/step - accuracy: 0.7687 - loss: 0.4765 - val_accuracy: 0.7273 - val_loss: 0.5161
Epoch 22/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7720 - loss: 0.4759 - val_accuracy: 0.7208 - val_loss: 0.5164
Epoch 23/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7606 - loss: 0.4752 - val_accuracy: 0.7143 - val_loss: 0.5165
Epoch 24/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7720 - loss: 0.4751 - val_accuracy: 0.7078 - val_loss: 0.5184
Epoch 25/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7671 - loss: 0.4769 - val_accuracy: 0.7013 - val_loss: 0.5163
Epoch 26/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7736 - loss: 0.4712 - val_accuracy: 0.7078 - val_loss: 0.5160
Epoch 27/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7850 - loss: 0.4640 - val_accuracy: 0.6948 - val_loss: 0.5161
Epoch 28/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7736 - loss: 0.4775 - val_accuracy: 0

***Tune the number of Neuron***

In [19]:
import shutil
def build_model_neurons(hp):
    # hp.Int samples an integer from [min_value, max_value] in increments of step
    units = hp.Int('units', min_value=8, max_value=128, step=16)

    model = Sequential([
        Dense(units=units, activation='relu', input_dim=8),
        Dropout(0.3),
        Dense(units=units // 2, activation='relu'),  # second layer is half the first
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


shutil.rmtree('tuner_neurons', ignore_errors=True)

tuner_neurons = kt.RandomSearch(
    build_model_neurons,
    objective='val_accuracy',
    max_trials=8,
    seed=42,
    directory='tuner_neurons',
    project_name='exp2'
)

tuner_neurons.search(
    X_train, y_train,
    epochs=20,
    validation_data=(X_test, y_test),
    callbacks=[EarlyStopping(monitor='val_loss', patience=5)],
    verbose=0
)

best_neurons_hp = tuner_neurons.get_best_hyperparameters()[0]
print(f"Best neuron config: {best_neurons_hp.values}")

Best neuron config: {'units': 72}


In [20]:
model_exp2 = tuner_neurons.get_best_models(num_models=1)[0]

history_exp2 = model_exp2.fit(
    X_train, y_train,
    epochs=100,
    validation_data=(X_test, y_test),
    callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
    initial_epoch=20,
    verbose=1
)

loss2, acc2 = model_exp2.evaluate(X_test, y_test, verbose=0)
print(f"Exp 2 (Neuron Tuning) Test Accuracy: {acc2:.4f}")

Epoch 21/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 100ms/step - accuracy: 0.7590 - loss: 0.4811 - val_accuracy: 0.7338 - val_loss: 0.4927
Epoch 22/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7720 - loss: 0.4719 - val_accuracy: 0.7273 - val_loss: 0.4915
Epoch 23/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7752 - loss: 0.4658 - val_accuracy: 0.7208 - val_loss: 0.4910
Epoch 24/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7769 - loss: 0.4476 - val_accuracy: 0.7208 - val_loss: 0.4929
Epoch 25/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7704 - loss: 0.4561 - val_accuracy: 0.7273 - val_loss: 0.4923
Epoch 26/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7964 - loss: 0.4485 - val_accuracy: 0.7273 - val_loss: 0.4909
Epoch 27/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7769 - loss: 0.4498 - val_accuracy: 0.7338 - val_loss: 0.4903
Epoch 28/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7818 - loss: 0.4439 - val_accuracy: 

***Tune number of hidden layer***

In [21]:
def build_model_layers(hp):
    num_layers = hp.Int('num_layers', min_value=1, max_value=6)

    model = Sequential()
    model.add(Dense(32, activation='relu', input_dim=8))

    for i in range(num_layers):
        model.add(Dense(units=32, activation='relu'))

        model.add(Dropout(0.2))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


shutil.rmtree('tuner_layers', ignore_errors=True)

tuner_layers = kt.RandomSearch(
    build_model_layers,
    objective='val_accuracy',
    max_trials=8,
    seed=42,
    directory='tuner_layers',
    project_name='exp3'
)

tuner_layers.search(
    X_train, y_train,
    epochs=20,
    validation_data=(X_test, y_test),
    callbacks=[EarlyStopping(monitor='val_loss', patience=5)],
    verbose=0
)

best_layers_hp = tuner_layers.get_best_hyperparameters()[0]
print(f"Best layer config: {best_layers_hp.values}")

Best layer config: {'num_layers': 4}


In [22]:
model_exp3 = tuner_layers.get_best_models(num_models=1)[0]

history_exp3 = model_exp3.fit(
    X_train, y_train,
    epochs=100,
    validation_data=(X_test, y_test),
    callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
    initial_epoch=20,
    verbose=1
)

loss3, acc3 = model_exp3.evaluate(X_test, y_test, verbose=0)
print(f"Exp 3 (Layer Tuning) Test Accuracy: {acc3:.4f}")

Epoch 21/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 167ms/step - accuracy: 0.7997 - loss: 0.4253 - val_accuracy: 0.7273 - val_loss: 0.5310
Epoch 22/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8046 - loss: 0.4166 - val_accuracy: 0.7273 - val_loss: 0.5333
Epoch 23/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8078 - loss: 0.4437 - val_accuracy: 0.7403 - val_loss: 0.5100
Epoch 24/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8111 - loss: 0.4158 - val_accuracy: 0.7532 - val_loss: 0.5109
Epoch 25/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8029 - loss: 0.4334 - val_accuracy: 0.7468 - val_loss: 0.5217
Epoch 26/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8046 - loss: 0.4158 - val_accuracy: 0.7468 - val_loss: 0.5131
Epoch 27/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8078 - loss: 0.4184 - val_accuracy: 0.7468 - val_loss: 0.5205
Epoch 28/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8176 - loss: 0.4135 - val_accuracy: 

# * Tune everything*

In [23]:
def build_model_full(hp):
    num_layers = hp.Int('num_layers', min_value=1, max_value=6)
    optimizer  = hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop'])
    dropout_rate = hp.Float('dropout_rate', min_value=0.1, max_value=0.5, step=0.1)

    model = Sequential()

    for i in range(num_layers):
        units = hp.Int(f'units_{i}', min_value=16, max_value=128, step=16)
        activation = hp.Choice(f'activation_{i}', values=['relu', 'tanh'])

        if i == 0:
            model.add(Dense(units=units, activation=activation, input_dim=8))
        else:
            model.add(Dense(units=units, activation=activation))

        model.add(Dropout(rate=dropout_rate))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


shutil.rmtree('tuner_full', ignore_errors=True)

tuner_full = kt.RandomSearch(
    build_model_full,
    objective='val_accuracy',
    max_trials=15,
    seed=42,
    directory='tuner_full',
    project_name='exp4'
)

tuner_full.search(
    X_train, y_train,
    epochs=20,
    validation_data=(X_test, y_test),
    callbacks=[EarlyStopping(monitor='val_loss', patience=5)],
    verbose=0
)

best_full_hp = tuner_full.get_best_hyperparameters()[0]
print(f"Best full config: {best_full_hp.values}")

Best full config: {'num_layers': 4, 'optimizer': 'adam', 'dropout_rate': 0.2, 'units_0': 64, 'activation_0': 'tanh', 'units_1': 64, 'activation_1': 'relu', 'units_2': 64, 'activation_2': 'relu', 'units_3': 16, 'activation_3': 'tanh'}


In [24]:
model_exp4 = tuner_full.get_best_models(num_models=1)[0]

history_exp4 = model_exp4.fit(
    X_train, y_train,
    epochs=100,
    validation_data=(X_test, y_test),
    callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
    initial_epoch=20,
    verbose=1
)

loss4, acc4 = model_exp4.evaluate(X_test, y_test, verbose=0)
print(f"Exp 4 (Full Tuning) Test Accuracy: {acc4:.4f}")

Epoch 21/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 147ms/step - accuracy: 0.7948 - loss: 0.4604 - val_accuracy: 0.7143 - val_loss: 0.5202
Epoch 22/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7818 - loss: 0.4723 - val_accuracy: 0.7208 - val_loss: 0.5147
Epoch 23/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7769 - loss: 0.4604 - val_accuracy: 0.7208 - val_loss: 0.5082
Epoch 24/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7818 - loss: 0.4444 - val_accuracy: 0.7208 - val_loss: 0.5123
Epoch 25/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7834 - loss: 0.4458 - val_accuracy: 0.7208 - val_loss: 0.5136
Epoch 26/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7915 - loss: 0.4371 - val_accuracy: 0.7208 - val_loss: 0.5145
Epoch 27/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7850 - loss: 0.4408 - val_accuracy: 0.7078 - val_loss: 0.5126
Epoch 28/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7899 - loss: 0.4395 - val_accuracy: 